In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import yaml
import os
from tqdm import tqdm

In [ ]:
from sklearn.preprocessing import scale, minmax_scale
from sklearn.metrics import root_mean_squared_error, ndcg_score
def calc_test(true_scores, pred_scores, k=10):
    rho, _ = stats.spearmanr(true_scores, pred_scores)

    # RMSE
    rmse = root_mean_squared_error(true_scores, pred_scores)

    # NDCG@k
    std_tgts = minmax_scale([true_scores], (0, 5), axis=1)
    ndcg_val = ndcg_score(std_tgts,[pred_scores], k=k)

    result ={
        'spearman': rho,
        'ndcg': ndcg_val
    }
    return result

In [ ]:
info_df = pd.read_csv("data/info.csv")
h3_dict = info_df.set_index("PDB")["CDRH3"].to_dict()
targets = list(h3_dict.keys())

In [ ]:
def get_ddg(path):
    results_df = pd.read_csv(path)
    ddg_scores = (results_df[results_df["scored_state"]=="ddG"]
                     .groupby("case_name")["total_score"]
                     .min()
                     .sort_index())
    return ddg_scores

def normalize_score(score):
    return (score-score.quantile(0.05))/(score.quantile(0.95)-score.quantile(0.05)+1e-10)

def ip_seq_objective(x, peak = 8, x_lower = 6.7, x_upper = 9.05):
    if x <= peak:
        return 0.5 - (0.5 / 1.3) * (x - x_lower)
    else:
        return (0.5 / (x_upper-peak)) * (x - peak)

In [ ]:
flex_ddg_dfs={}
sampled_seq_dfs = {}
flex_ddg_df_alls = {}
for target in targets:
    for mode in ["bias"]:
        flex_ddg_df = pd.read_csv(f"flexddg_online/flexddgs/{target}/{mode}/outputs-results.csv")
        flex_ddg_df = flex_ddg_df[flex_ddg_df["scored_state"]=="ddG"].groupby("case_name")["total_score"].min().sort_index()
        flex_ddg_dfs[target+"_"+mode]=flex_ddg_df
        sampled_seq_dfs[target+"_"+mode]=pd.read_csv(f"flexddg_online/flexddgs/{target}/{mode}/sampled_mutations.csv", index_col=0)
        
test_dfs = {target: pd.read_csv(f"flexddg_online/flexddgs/{target}/bias/sampled_mutations.csv") for target in targets}
for target in test_dfs:
    test_dfs[target]["DMS_score"] = - flex_ddg_dfs[target+"_bias"].values

In [ ]:
jobdf = pd.read_csv(jobfile)
cycles = {}
dfs = []
configs = []
ref_points = []
conf_datas = []
score_cols = ["acquisition_score", "ablang2_perplexity", "IP_seq", "instability_index", "hydrophobicity"]
score_cols_std = [col+"_std" for col in score_cols]
for confpath in tqdm(jobdf["CONFIG"]):
    try:
        with open(confpath) as f:
            data = yaml.safe_load(f)
        target=data["data_dir"].split("/")[1]
        model_type = data["data_dir"].split("/")[2]
        exp=data["data_dir"].split("/")[3]
        df = pd.read_csv(os.path.join(data_dir, "..", data["data_dir"], "9", "train_data", "training_data.csv"))
        df["target"]=target
        df["model_type"]=model_type
        df["mutations"] = df["mutations"].fillna("")
        df["exp"]=exp
        df["flxddg"] = -df["DMS_score"]
    
        df["flxddg_std"] = normalize_score(df["flxddg"])
        df["acquisition_score_std"] = df["flxddg_std"]
        df["ablang2_perplexity_std"] = normalize_score(df["ablang2_perplexity"])
        df["IP_seq_std"] = normalize_score(df["IP_seq"].apply(ip_seq_objective))
        df["instability_index_std"] = normalize_score(df["instability_index"])
        df["hydrophobicity_std"] = normalize_score(df["hydrophobicity"])
        acquisition_weight = data.get("acquisition_weight", {"acquisition_score": 2})
        ref_point = {}
        for col in score_cols:
            w = acquisition_weight.get(col, 0)
            df[col+"_std"] = df[col+"_std"]*w
            if w>0:
                ref_point[col+"_std"] = w
        ref_points.append(ref_point)
        df["sum_score"] = df[score_cols_std].sum(axis=1)
        
        df["#Mutation"]=df["mutations"].apply(lambda x: len(x.split(",")) if x !="" else 0)
        dfs.append(df)
        configs.append({
            "target":target,
            "MAXCYCLE":10,
            "model_type": model_type,
            "exp":exp,
            "data_dir": data["data_dir"]
        })
        ddgs_list = []
        for cycle in range(10):
            ddgs = get_ddg(os.path.join(data_dir, "..", data["data_dir"], str(cycle), "flex_ddG", "outputs-results.csv"))
            ddgs = ddgs.reset_index()
            ddgs["cycle"]=cycle
            ddgs_list.append(ddgs)
        conf_datas.append(data)
    except Exception as e:
        print("Failed", confpath, e)
        continue

In [ ]:
import yaml
from tqdm import tqdm

def hamming_distance(seq1, seq2):
    """Calculate Hamming distance between two sequences"""
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

def select_diverse_subset(df, N, score_col, ascending=False):
    """Select N diverse sequences based on score, ensuring Hamming distance > 1"""
    df_sorted = df.sort_values(score_col, ascending=ascending)
    selected_indices = []
    
    for idx in df_sorted.index:
        if len(selected_indices) >= N:
            break
            
        current_seq = df.loc[idx, "mutseq"]
        is_diverse = True
        
        for selected_idx in selected_indices:
            selected_seq = df.loc[selected_idx, "mutseq"]
            if hamming_distance(current_seq, selected_seq) <= 1:
                is_diverse = False
                break
        
        if is_diverse:
            selected_indices.append(idx)
    
    return df.loc[selected_indices]

N=40

all_df_merges=[]
top_df_merges=[]
sum_df_merges=[]
for i in range(len(dfs)):
    target=configs[i]["target"]
    exp=configs[i]["exp"]
    df = dfs[i]
    conf_data = conf_datas[i]
    acquisition_weight = conf_data.get("acquisition_weight", {})
    
    CYCLE=configs[i]["MAXCYCLE"]
    df_ = df[df["flxddg"]>-100].copy()
    top_dfs = {}
    for cycle in range(CYCLE):
        cycle_df = df_[df_["cycle"]<=cycle]
        top_dfs[cycle+1] = select_diverse_subset(cycle_df, N, "DMS_score", ascending=False)
    
    all_dfs = {cycle+1: df_[df_["cycle"]<=cycle] for cycle in range(CYCLE)}
    df_c = df_.copy()
    ref_point = ref_points[i]
    for ref_col, ref_p in ref_point.items():
        df_c = df_c[df_c[ref_col] <= ref_p]
    top1_dfs = {cycle+1: df[df["cycle"]<=cycle].sort_values("DMS_score", ascending=False).head(5)
               for cycle in range(CYCLE)}

    sum_dfs = {}
    for cycle in range(CYCLE):
        cycle_df = df_[df_["cycle"]<=cycle]
        sum_dfs[cycle+1] = select_diverse_subset(cycle_df, N, "sum_score", ascending=True)
    
    sum_df_merge = pd.concat(sum_dfs)
    top_df_merge = pd.concat(top_dfs)
    all_df_merge = pd.concat(all_dfs)
    top_df_merge.index.names=["CYCLE", "index"]
    all_df_merge.index.names=["CYCLE", "index"]
    sum_df_merge.index.names=["CYCLE", "index"]
    top_df_merge = top_df_merge.reset_index()
    all_df_merge = all_df_merge.reset_index()
    sum_df_merge = sum_df_merge.reset_index()
    top_df_merges.append(top_df_merge)
    all_df_merges.append(all_df_merge)
    sum_df_merges.append(sum_df_merge)
top_df_merge_cat = pd.concat(top_df_merges)
all_df_merge_cat = pd.concat(all_df_merges)
sum_df_merge_cat = pd.concat(sum_df_merges)


In [ ]:
all_test_scores=[]
for conf in configs:
    target = conf["target"]
    if target not in targets:
        continue
    for cycle in range(10):
        input_dir = os.path.join(data_dir, conf["target"], conf["model_type"], conf["exp"], str(cycle), "train_data")
        test_pred = np.load(os.path.join(input_dir, "test_inference_bias.npy"))
        test_df = test_dfs[target]
        test_df_ = test_df.copy()
        test_df_["Pred"] = test_pred
        all_test_scores.append({
            **calc_test(test_df_["DMS_score"], test_df_["Pred"]),
            "CYCLE": cycle+1,
            "target": conf["target"],
            "model_type": conf["model_type"],
            "exp": conf["exp"],
        })
all_test_scores_cat = pd.DataFrame(all_test_scores)
all_test_scores_cat["spearman"] = all_test_scores_cat["spearman"].fillna(0)

In [ ]:
sum_df_merge_cat["EXP"] = sum_df_merge_cat["model_type"]+"+"+sum_df_merge_cat["exp"]
top_df_merge_cat["EXP"] = top_df_merge_cat["model_type"]+"+"+top_df_merge_cat["exp"]
all_df_merge_cat["EXP"] = all_df_merge_cat["model_type"]+"+"+all_df_merge_cat["exp"]
all_test_scores_cat["EXP"] = all_test_scores_cat["model_type"]+"+"+all_test_scores_cat["exp"]

In [ ]:
sum_df_merge_cat_abo.to_csv("results/flexddg_online/single/sum_results.csv",index=False)
top_df_merge_cat_abo.to_csv("results/flexddg_online/single/top_results.csv",index=False)
all_df_merge_cat_abo.to_csv("results/flexddg_online/single/all_results.csv",index=False)
all_test_scores_cat_abo.to_csv("results/flexddg_online/single/all_results_test.csv",index=False)